In [1]:
%pip install tensorflow numpy matplotlib

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Load MNIST
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Flatten for dense-only models
x_train_flat = x_train.reshape(-1, 28*28)
x_test_flat = x_test.reshape(-1, 28*28)

# One-hot encode labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


2026-03-02 02:14:39.729838: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-02 02:14:58.448454: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-02 02:15:10.135735: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


# Question 1 – No Hidden Layers (Logistic Regression Model)

In [2]:
model_q1 = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(10, activation="softmax")
])

model_q1.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_q1 = model_q1.fit(
    x_train_flat, y_train_cat,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

test_loss_q1, test_acc_q1 = model_q1.evaluate(x_test_flat, y_test_cat)

2026-03-02 02:15:14.775284: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/10


2026-03-02 02:15:15.157698: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 169344000 exceeds 10% of free system memory.


422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8284 - loss: 0.7116 - val_accuracy: 0.9122 - val_loss: 0.3515
Epoch 2/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8983 - loss: 0.3785 - val_accuracy: 0.9257 - val_loss: 0.2868
Epoch 3/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9084 - loss: 0.3322 - val_accuracy: 0.9282 - val_loss: 0.2633
Epoch 4/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9145 - loss: 0.3105 - val_accuracy: 0.9305 - val_loss: 0.2519
Epoch 5/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9172 - loss: 0.2972 - val_accuracy: 0.9355 - val_loss: 0.2437
Epoch 6/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9194 - loss: 0.2887 - val_accuracy: 0.9367 - val_loss: 0.2386
Epoch 7/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9210 - loss: 0.2822 - val_accuracy: 0.9352 - val_loss: 0.2330
Epoch 8/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9228 - loss: 0.2767 - val_accuracy: 0.9358 - val_

In this section, I trained a neural network with no hidden layers. The model directly maps the 784 input features to 10 output neurons using a softmax activation. This is essentially multiclass logistic regression.

The model achieved around 93 % accuracy on the test. This makes sense because without hidden layers, the model is linear. It can only learn linear decision boundaries between classes. However, digits are not perfectly linearly separable, so performance is limited.

I experimented with increasing the number of epochs, but accuracy did not improve significantly after about 10 epochs. That suggests the model had already converged. Changing the learning rate too much caused instability, while adjusting batch size mainly affected training speed rather than final accuracy.

Overall, this model performs reasonably well but has structural limitations.

# Question 2 – Backpropagation From Scratch (One Hidden Layer)

In [3]:
subset_size = 5000
X = x_train_flat[:subset_size]
Y = y_train_cat[:subset_size]

input_size = 784
hidden_size = 64
output_size = 10
learning_rate = 0.5   # increased
epochs = 50           # increased

# Better initialization (He initialization for ReLU)
W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2 / input_size)
b1 = np.zeros((1, hidden_size))
W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2 / hidden_size)
b2 = np.zeros((1, output_size))

def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

def softmax(z):
    exp = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)

def cross_entropy(y_true, y_pred):
    return -np.mean(np.sum(y_true * np.log(y_pred + 1e-8), axis=1))

for epoch in range(epochs):

    # Forward pass
    Z1 = X @ W1 + b1
    A1 = relu(Z1)
    Z2 = A1 @ W2 + b2
    A2 = softmax(Z2)

    loss = cross_entropy(Y, A2)

    # Backprop
    dZ2 = A2 - Y
    dW2 = A1.T @ dZ2 / subset_size
    db2 = np.sum(dZ2, axis=0, keepdims=True) / subset_size

    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * relu_deriv(Z1)
    dW1 = X.T @ dZ1 / subset_size
    db1 = np.sum(dZ1, axis=0, keepdims=True) / subset_size

    # Update
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    if epoch % 5 == 0:
        print("Epoch:", epoch, "Loss:", loss)

Epoch: 0 Loss: 2.3947419577952744
Epoch: 5 Loss: 1.8832689103775935
Epoch: 10 Loss: 0.7526258876358447
Epoch: 15 Loss: 0.6807235158847393
Epoch: 20 Loss: 0.5661783474866477
Epoch: 25 Loss: 0.4451866328550067
Epoch: 30 Loss: 0.40739014577721216
Epoch: 35 Loss: 0.6143611777207201
Epoch: 40 Loss: 0.3387970105434169
Epoch: 45 Loss: 0.31007947928773455


In this section, I implemented a fully connected neural network with one hidden layer without using Keras training functions. The goal was to manually implement the forward pass, compute the loss, and update weights using backpropagation.

Forward pass:
First, the input matrix is multiplied by the first wieght matrix and the bias is added. Then a ReLU activation is applied to introduce nonlinearity. Then, the hidden activations are multiplied by the second weight matrix and bias to produce output values. Finally, softmax is applied to convert those values into probabilities across the 10 digit classes.

Loss:
This measures how far the predicted probability distribution is from the true distribution. If the model is confident and wrong, the loss increases a lot. If it is confident and correct, the loss is small.

Backpropagation:
Backpropagation works by computing how the loss changes with respect to each weight. I first compute the error at the output layer by subtracting the true labels from the predicted probabilities. Then I propagate that error backward through the second layer to compute gradients for its weights and biases ('back propagation'... you get it?). After that, I pass the gradient through the ReLU derivative and compute gradients for the first layer.

Then, I update each weight and bias by subtracting the learning rate times its gradient.

After fixing initialization and learning rate, the loss decreased from about 2.4, which corresponds to random guessing, down to around 0.31. There was a slight bump at one point, but overall the trend was clearly downward. This confirms that my implementation of the chain rule and gradient descent is working correctly.

# Question 3 – Fully Connected Deep Network (No Convolution)

In [4]:
model_q3 = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(16, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(16, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model_q3.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_q3 = model_q3.fit(
    x_train_flat, y_train_cat,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

test_loss_q3, test_acc_q3 = model_q3.evaluate(x_test_flat, y_test_cat)

Epoch 1/15


2026-03-02 02:15:28.946121: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 169344000 exceeds 10% of free system memory.


422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7651 - loss: 0.8300 - val_accuracy: 0.9217 - val_loss: 0.2940
Epoch 2/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8795 - loss: 0.4131 - val_accuracy: 0.9357 - val_loss: 0.2266
Epoch 3/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8959 - loss: 0.3527 - val_accuracy: 0.9420 - val_loss: 0.2042
Epoch 4/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9016 - loss: 0.3276 - val_accuracy: 0.9453 - val_loss: 0.1901
Epoch 5/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9044 - loss: 0.3141 - val_accuracy: 0.9497 - val_loss: 0.1738
Epoch 6/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9093 - loss: 0.3006 - val_accuracy: 0.9523 - val_loss: 0.1674
Epoch 7/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9133 - loss: 0.2862 - val_accuracy: 0.9525 - val_loss: 0.1614
Epoch 8/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9166 - loss: 0.2736 - val_accuracy: 0.9542 - val_

In this section, I added two hidden layers with 16 neurons each. I also included batch normalization and dropout.

Adding hidden layers increases the expressive power of the network. Unlike the model in Question 1, this network can now learn nonlinear decision boundaries. This allows it to better distinguish between digits that might look similar but differ in subtle ways.

Batch normalization helps stabilize training by keeping activations in a reasonable range. This improves gradient flow and makes training more consistent across epochs.

Dropout randomly turns off some neurons during training. This reduces overfitting and forces the network to learn more robust features instead of relying too heavily on specific neurons.

The test accuracy improved compared to the no hidden layer model, reaching around 95 percent. The improvement was not dramatic, but it confirms that depth increases representational capacity. Increasing the number of layers further did not increase accuracy by more than a few percent, which suggests dmiminshing returns for purely dense networks on MNIST.

# Question 4 – Convolutional Neural Network with MaxPooling

In [5]:
# Reshape for CNN
x_train_cnn = x_train.reshape(-1, 28, 28, 1)
x_test_cnn = x_test.reshape(-1, 28, 28, 1)

model_q4 = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(16, (3,3), activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(32, (3,3), activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(32, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model_q4.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_q4 = model_q4.fit(
    x_train_cnn, y_train_cat,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

test_loss_q4, test_acc_q4 = model_q4.evaluate(x_test_cnn, y_test_cat)

Epoch 1/10


2026-03-02 02:15:45.773167: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 169344000 exceeds 10% of free system memory.


422/422 ━━━━━━━━━━━━━━━━━━━━ 14s 30ms/step - accuracy: 0.8970 - loss: 0.3531 - val_accuracy: 0.9727 - val_loss: 0.0965
Epoch 2/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 20s 29ms/step - accuracy: 0.9693 - loss: 0.1015 - val_accuracy: 0.9787 - val_loss: 0.0755
Epoch 3/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 12s 29ms/step - accuracy: 0.9773 - loss: 0.0741 - val_accuracy: 0.9815 - val_loss: 0.0629
Epoch 4/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 21s 30ms/step - accuracy: 0.9809 - loss: 0.0608 - val_accuracy: 0.9838 - val_loss: 0.0563
Epoch 5/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 21s 30ms/step - accuracy: 0.9847 - loss: 0.0502 - val_accuracy: 0.9842 - val_loss: 0.0569
Epoch 6/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 20s 29ms/step - accuracy: 0.9870 - loss: 0.0434 - val_accuracy: 0.9858 - val_loss: 0.0496
Epoch 7/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 12s 29ms/step - accuracy: 0.9883 - loss: 0.0387 - val_accuracy: 0.9873 - val_loss: 0.0440
Epoch 8/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 20s 29ms/step - accuracy: 0.9896 - loss: 0.0337 - val_accurac

In this section, I used convolutional layers instead of only dense layers.

Convolutional layers are suited for image recognition due to the fact that they preserve spatial structure. Instead of treating the image as a flat vector, convolutional filters scan across the image and detect local patterns such as edges, curves, and strokes. These low level features combine into higher level features in deeper layers.

MaxPooling reduces the spatial size of the feature maps by keeping only the strongest activation in each region. This reduces computation and helps the model become more invariant to small shifts in the image. It also reduces overfitting by lwering the number of parammeters.

As expected, the convolutional network significantly outperformed the dense models, with nearly 99 percent accuracy. Even though each epoch took longer, fewer parameters were needed in the fully connected part of the network, and the overall model was more efficient and more accurate.

# Question 5 (Bonus) – Improved CNN (Under 15 Epochs) .... FYI, this take 15 minutes to run

In [6]:
model_q5 = keras.Sequential([
    layers.Input(shape=(28,28,1)),
    layers.Conv2D(32, (3,3), activation="relu"),
    layers.Conv2D(32, (3,3), activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),
    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax")
])

model_q5.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_q5 = model_q5.fit(
    x_train_cnn, y_train_cat,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

test_loss_q5, test_acc_q5 = model_q5.evaluate(x_test_cnn, y_test_cat)

Epoch 1/15


2026-03-02 02:18:40.521753: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 169344000 exceeds 10% of free system memory.


422/422 ━━━━━━━━━━━━━━━━━━━━ 59s 136ms/step - accuracy: 0.8142 - loss: 0.5773 - val_accuracy: 0.9772 - val_loss: 0.0823
Epoch 2/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 82s 138ms/step - accuracy: 0.9410 - loss: 0.2026 - val_accuracy: 0.9822 - val_loss: 0.0586
Epoch 3/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 56s 133ms/step - accuracy: 0.9535 - loss: 0.1552 - val_accuracy: 0.9845 - val_loss: 0.0481
Epoch 4/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 57s 135ms/step - accuracy: 0.9619 - loss: 0.1311 - val_accuracy: 0.9880 - val_loss: 0.0410
Epoch 5/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 55s 131ms/step - accuracy: 0.9665 - loss: 0.1149 - val_accuracy: 0.9882 - val_loss: 0.0416
Epoch 6/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 57s 134ms/step - accuracy: 0.9702 - loss: 0.1017 - val_accuracy: 0.9892 - val_loss: 0.0365
Epoch 7/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 57s 135ms/step - accuracy: 0.9733 - loss: 0.0913 - val_accuracy: 0.9897 - val_loss: 0.0355
Epoch 8/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 82s 134ms/step - accuracy: 0.9758 - loss: 0.0807 - val

For the bonus, I experimented with a deeper convolutional network whilst keeping the number of epochs under 15. I added additional convolutional layers and dropout for regularization.

This model achieved over 99.3 percent accuracy at max, which exceeds the baseline Keras example. The deeper structure should allow the network to learn more abstract features, starting from edges and small shapes and building up to full digit representations.

However, the deeper network also required more training time per epoch. In terms of speed to accuracy tradeoff, the moderate CNN from Question 4 was probably the most efficient. The bonus model achieved slightly higher accuracy but at a significant computational cost (yes... this takes a long time).

Overall, this demonstrates that convolutional networks are significantly more effective than fully connected networks for image classification tasks.